In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import pandas as pd

df = pd.read_csv(r"C:\Users\BHIMA SHANKAR\Downloads\agrisense_advanced_dataset.csv")   # load dataset
df = df.sort_values(by='Date')

In [18]:
features = ['Temperature', 'Humidity', 'Rainfall']

In [19]:
X, y_disease, y_severity = [], [], []

for i in range(len(df) - time_steps):
    X.append(df.iloc[i:i+time_steps][features].values)
    y_disease.append(df.iloc[i+time_steps]['Disease'])
    y_severity.append(df.iloc[i+time_steps]['Severity(%)'])

In [25]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import LSTM, Dense, Input
import numpy as np
from sklearn.preprocessing import LabelEncoder

# Encode labels
le = LabelEncoder()
y_disease = le.fit_transform(y_disease)

num_features = len(features)
num_classes = len(set(y_disease))

# Convert to numpy
X = np.array(X)
y_disease = np.array(y_disease)
y_severity = np.array(y_severity)

# Model
input_layer = Input(shape=(7, num_features))
x = LSTM(64)(input_layer)

disease_output = Dense(num_classes, activation='softmax', name='disease')(x)
severity_output = Dense(1, activation='linear', name='severity')(x)

model = Model(inputs=input_layer, outputs=[disease_output, severity_output])

model.compile(
    optimizer='adam',
    loss={
        'disease': 'sparse_categorical_crossentropy',
        'severity': 'mse'
    },
    metrics={
        'disease': 'accuracy',
        'severity': 'mae'
    }
)

model.fit(
    X,
    {'disease': y_disease, 'severity': y_severity},
    epochs=10,
    batch_size=32,
    validation_split=0.2
)

Epoch 1/10
180/180 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - disease_accuracy: 0.3876 - disease_loss: 2.3578 - loss: 1679.9766 - severity_loss: 1677.9248 - severity_mae: 30.5469 - val_disease_accuracy: 0.4496 - val_disease_loss: 2.2354 - val_loss: 1434.6969 - val_severity_loss: 1432.0424 - val_severity_mae: 30.1309
Epoch 2/10
180/180 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - disease_accuracy: 0.4482 - disease_loss: 2.2299 - loss: 1266.0553 - severity_loss: 1263.7117 - severity_mae: 29.5119 - val_disease_accuracy: 0.4496 - val_disease_loss: 2.2176 - val_loss: 1236.4662 - val_severity_loss: 1234.0103 - val_severity_mae: 29.8115
Epoch 3/10
180/180 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - disease_accuracy: 0.4482 - disease_loss: 2.2235 - loss: 1148.4771 - severity_loss: 1145.9491 - severity_mae: 29.0362 - val_disease_accuracy: 0.4496 - val_disease_loss: 2.1712 - val_loss: 1150.0919 - val_severity_loss: 1147.7548 - val_severity_mae: 28.5579
Epoch 4/10
180/180 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - disease_accuracy

In [27]:
import pandas as pd
import numpy as np

# Load dataset
df = pd.read_csv(r"C:\Users\BHIMA SHANKAR\Downloads\agrisense_advanced_dataset.csv")

# Convert datetime
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values(by='Date')

# Encode categorical
from sklearn.preprocessing import LabelEncoder

le_crop = LabelEncoder()
le_region = LabelEncoder()
le_disease = LabelEncoder()

df['Crop'] = le_crop.fit_transform(df['Crop'])
df['Region'] = le_region.fit_transform(df['Region'])
df['Disease'] = le_disease.fit_transform(df['Disease'])

# Features
features = ['Crop', 'Region', 'Temperature', 'Humidity', 
            'Rainfall', 'SoilMoisture', 'CropStage']

target_disease = 'Disease'
target_severity = 'Severity(%)'

In [28]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
df[features] = scaler.fit_transform(df[features])

In [29]:
time_steps = 7

X, y_disease, y_severity = [], [], []

for i in range(len(df) - time_steps):
    X.append(df.iloc[i:i+time_steps][features].values)
    y_disease.append(df.iloc[i+time_steps][target_disease])
    y_severity.append(df.iloc[i+time_steps][target_severity])

X = np.array(X)
y_disease = np.array(y_disease)
y_severity = np.array(y_severity)

In [30]:
split = int(0.8 * len(X))

X_train, X_test = X[:split], X[split:]
y_d_train, y_d_test = y_disease[:split], y_disease[split:]
y_s_train, y_s_test = y_severity[:split], y_severity[split:]

In [31]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import LSTM, Dense, Input

input_layer = Input(shape=(time_steps, len(features)))

x = LSTM(64, return_sequences=False)(input_layer)

# Outputs
disease_output = Dense(len(np.unique(y_disease)), activation='softmax')(x)
severity_output = Dense(1, activation='linear')(x)

model = Model(inputs=input_layer, outputs=[disease_output, severity_output])

model.compile(
    optimizer='adam',
    loss=['sparse_categorical_crossentropy', 'mse'],
    metrics=['accuracy']
)

model.summary()

Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_7       │ (None, 7, 7)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_3 (LSTM)       │ (None, 64)        │     18,432 │ input_layer_7[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 17)        │      1,105 │ lstm_3[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 1)         │         65 │ lstm_3[0][0]      │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 19,602 (76.57 KB)

 Trainable params: 19,602 (76.57 KB)

 Non-trainable params: 0 (0.00 B)

In [44]:
model.compile(
    optimizer='adam',
    loss=['sparse_categorical_crossentropy', 'mse'],
    metrics=[['accuracy'], ['mae']]
)

model.fit(
    X_train,
    [y_d_train, y_s_train],
    epochs=10,
    batch_size=32,
    validation_data=(X_test, [y_d_test, y_s_test])
)

Epoch 1/10
180/180 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - dense_1_loss: 1484.0302 - dense_1_mae: 30.0089 - dense_accuracy: 0.3650 - dense_loss: 2.3657 - loss: 1486.9274 - val_dense_1_loss: 1280.4340 - val_dense_1_mae: 29.8161 - val_dense_accuracy: 0.4496 - val_dense_loss: 2.2220 - val_loss: 1282.9500
Epoch 2/10
180/180 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - dense_1_loss: 1177.5752 - dense_1_mae: 29.4606 - dense_accuracy: 0.4482 - dense_loss: 2.2281 - loss: 1179.7671 - val_dense_1_loss: 1186.1591 - val_dense_1_mae: 29.9581 - val_dense_accuracy: 0.4496 - val_dense_loss: 2.2217 - val_loss: 1188.5330
Epoch 3/10
180/180 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - dense_1_loss: 1134.9017 - dense_1_mae: 29.7471 - dense_accuracy: 0.4482 - dense_loss: 2.2264 - loss: 1137.3136 - val_dense_1_loss: 1171.0698 - val_dense_1_mae: 30.1733 - val_dense_accuracy: 0.4496 - val_dense_loss: 2.2194 - val_loss: 1173.3821
Epoch 4/10
180/180 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - dense_1_loss: 1129.6145 - dense_1_mae: 29.8976 - den

In [45]:
model.save("crop_disease_lstm_model.h5")

In [46]:
last_input = df[features].values[-time_steps:]
current_input = last_input.reshape(1, time_steps, len(features))

In [47]:
future_predictions = []

for i in range(7):
    pred_disease, pred_severity = model.predict(current_input)

    disease_class = np.argmax(pred_disease)
    severity_value = pred_severity[0][0]

    future_predictions.append((disease_class, severity_value))

    # Create next input (use predicted values)
    next_step = current_input[0][-1].copy()
    
    # Update (you can improve this with real forecasted weather)
    next_step[0] = next_step[0]  # Crop
    next_step[1] = next_step[1]  # Region
    
    current_input = np.append(current_input[:,1:,:], [[next_step]], axis=1)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 132ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step


In [48]:
results = []

for i, (d, s) in enumerate(future_predictions):
    results.append({
        "Day": i+1,
        "Predicted Disease": le_disease.inverse_transform([d])[0],
        "Predicted Severity (%)": round(float(s), 2)
    })

forecast_df = pd.DataFrame(results)
print(forecast_df)

   Day Predicted Disease  Predicted Severity (%)
0    1           Healthy                   19.27
1    2           Healthy                   19.51
2    3           Healthy                   19.01
3    4           Healthy                   19.49
4    5           Healthy                   19.05
5    6           Healthy                   19.70
6    7           Healthy                   19.06
